# 06 - Deep Learning Models: LSTM & Conv1D-LSTM for AQI Prediction

Time-series deep learning models using sliding window sequences:

| Model | Architecture | Purpose |
|-------|-------------|--------|
| Vanilla LSTM | LSTM → Dense | Baseline temporal model |
| Stacked LSTM | 2×LSTM → Dense | Deeper temporal patterns |
| Bidirectional LSTM | BiLSTM → Dense | Forward + backward context |
| Conv1D-LSTM | Conv1D → LSTM → Dense | Local patterns + temporal |
| GRU | GRU → Dense | Lightweight alternative |

**Key concept**: We reshape tabular data into 3D sequences (samples, timesteps, features) using a sliding window of past days to predict the next day's AQI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
import sys, os
import joblib

sys.path.insert(0, os.path.abspath('..'))
from config import DATA_PROCESSED, FIGURES_DIR, MODELS_DIR, RANDOM_STATE
from src.evaluation.metrics import regression_metrics

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
os.makedirs(MODELS_DIR / 'deep_learning', exist_ok=True)

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f'TensorFlow version: {tf.__version__}')

## 1. Load Data & Create Sequences

For LSTM models, we need 3D input: `(samples, timesteps, features)`. We use a **sliding window** approach:
- Look back `WINDOW_SIZE` days of pollutant readings
- Predict the AQI on the next day

We use only the core pollutant features (not lag/rolling features, since the LSTM learns its own temporal patterns from the sequence).

In [ ]:
# Load the cleaned data (pre-feature engineering) for sequence creation
df_clean = pd.read_parquet(DATA_PROCESSED / 'city_day_clean.parquet')
print(f"Clean data: {df_clean.shape}")
print(f"Cities: {df_clean['City'].nunique()}")
print(f"Date range: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}")

# Core pollutant features for sequences (no engineered features - LSTM learns its own patterns)
seq_features = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'NH3']
seq_features = [f for f in seq_features if f in df_clean.columns]
target = 'AQI'

# Drop rows with missing values in our feature set
df_seq = df_clean[['City', 'Date'] + seq_features + [target]].dropna().reset_index(drop=True)
print(f"After dropping NaN: {df_seq.shape}")
print(f"\nFeatures for sequences: {seq_features}")

In [ ]:
WINDOW_SIZE = 14  # 14 days lookback

def create_sequences(df, feature_cols, target_col, window_size, city_col='City'):
    """Create sliding window sequences per city (no cross-city leakage)."""
    X_list, y_list = [], []
    
    for city in df[city_col].unique():
        city_data = df[df[city_col] == city].sort_values('Date')
        features = city_data[feature_cols].values
        targets = city_data[target_col].values
        
        for i in range(window_size, len(city_data)):
            X_list.append(features[i - window_size:i])
            y_list.append(targets[i])
    
    return np.array(X_list), np.array(y_list)

# Temporal split: same proportions as main pipeline
df_seq = df_seq.sort_values('Date').reset_index(drop=True)
n = len(df_seq)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

df_train_seq = df_seq.iloc[:train_end]
df_val_seq = df_seq.iloc[train_end:val_end]
df_test_seq = df_seq.iloc[val_end:]

print(f"Train: {len(df_train_seq)}, Val: {len(df_val_seq)}, Test: {len(df_test_seq)}")
print(f"Train dates: {df_train_seq['Date'].min().date()} to {df_train_seq['Date'].max().date()}")
print(f"Test dates:  {df_test_seq['Date'].min().date()} to {df_test_seq['Date'].max().date()}")

In [ ]:
# Normalize features using training statistics only
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_train_seq[seq_features] = scaler.fit_transform(df_train_seq[seq_features])
df_val_seq[seq_features] = scaler.transform(df_val_seq[seq_features])
df_test_seq[seq_features] = scaler.transform(df_test_seq[seq_features])

# Create sequences
X_train, y_train = create_sequences(df_train_seq, seq_features, target, WINDOW_SIZE)
X_val, y_val = create_sequences(df_val_seq, seq_features, target, WINDOW_SIZE)
X_test, y_test = create_sequences(df_test_seq, seq_features, target, WINDOW_SIZE)

print(f"X_train: {X_train.shape} (samples, timesteps={WINDOW_SIZE}, features={len(seq_features)})")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_test range: [{y_test.min():.0f}, {y_test.max():.0f}], mean: {y_test.mean():.1f}")

## 2. Define Deep Learning Models

In [ ]:
n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]

def build_vanilla_lstm():
    model = keras.Sequential([
        layers.Input(shape=(n_timesteps, n_features)),
        layers.LSTM(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name='Vanilla_LSTM')
    return model

def build_stacked_lstm():
    model = keras.Sequential([
        layers.Input(shape=(n_timesteps, n_features)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name='Stacked_LSTM')
    return model

def build_bidirectional_lstm():
    model = keras.Sequential([
        layers.Input(shape=(n_timesteps, n_features)),
        layers.Bidirectional(layers.LSTM(64)),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name='Bidirectional_LSTM')
    return model

def build_conv1d_lstm():
    model = keras.Sequential([
        layers.Input(shape=(n_timesteps, n_features)),
        layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2),
        layers.LSTM(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name='Conv1D_LSTM')
    return model

def build_gru():
    model = keras.Sequential([
        layers.Input(shape=(n_timesteps, n_features)),
        layers.GRU(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name='GRU')
    return model

model_builders = {
    'Vanilla LSTM': build_vanilla_lstm,
    'Stacked LSTM': build_stacked_lstm,
    'Bidirectional LSTM': build_bidirectional_lstm,
    'Conv1D-LSTM': build_conv1d_lstm,
    'GRU': build_gru,
}

# Print architecture of each
for name, builder in model_builders.items():
    m = builder()
    print(f"\n{name}: {m.count_params():,} parameters")

## 3. Train All Models

In [ ]:
EPOCHS = 100
BATCH_SIZE = 32

dl_results = {}
dl_predictions = {}
dl_histories = {}

for name, builder in model_builders.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print(f"{'='*60}")
    
    model = builder()
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='mse', metrics=['mae'])
    
    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
    )
    
    start = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )
    elapsed = time.time() - start
    dl_histories[name] = history
    
    # Predict
    y_pred = model.predict(X_test, verbose=0).flatten()
    dl_predictions[name] = y_pred
    
    # Metrics
    metrics = regression_metrics(y_test, y_pred)
    metrics['Train_Time_s'] = round(elapsed, 2)
    metrics['Epochs'] = len(history.history['loss'])
    dl_results[name] = metrics
    
    print(f"  R2={metrics['R2']:.4f}, RMSE={metrics['RMSE']:.2f}, "
          f"MAE={metrics['MAE']:.2f}, Epochs={metrics['Epochs']}, Time={elapsed:.1f}s")
    
    # Save model
    model.save(MODELS_DIR / 'deep_learning' / f'{name.lower().replace(" ", "_").replace("-", "_")}.keras')

print("\nAll deep learning models trained and saved.")

## 4. Model Comparison

In [ ]:
# Results table
dl_df = pd.DataFrame(dl_results).T.sort_values('R2', ascending=False)

print("=" * 80)
print("DEEP LEARNING MODEL COMPARISON (Test Set)")
print("=" * 80)
print(dl_df.round(4).to_string())

# Comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

models_sorted = dl_df.index.tolist()
colors = ['#9C27B0' if 'Conv' in m else '#2196F3' if 'LSTM' in m else '#FF9800' for m in models_sorted]

axes[0].barh(models_sorted, dl_df['R2'], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_xlabel('R² Score')
axes[0].set_title('R² Score (higher is better)')
for i, v in enumerate(dl_df['R2']):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

axes[1].barh(models_sorted, dl_df['RMSE'], color=colors, edgecolor='black', alpha=0.85)
axes[1].set_xlabel('RMSE')
axes[1].set_title('RMSE (lower is better)')

axes[2].barh(models_sorted, dl_df['MAE'], color=colors, edgecolor='black', alpha=0.85)
axes[2].set_xlabel('MAE')
axes[2].set_title('MAE (lower is better)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '21_dl_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Training History

In [ ]:
# Training curves for all models
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (name, history) in enumerate(dl_histories.items()):
    if i >= len(axes):
        break
    ax = axes[i]
    ax.plot(history.history['loss'], label='Train Loss', linewidth=1.5)
    ax.plot(history.history['val_loss'], label='Val Loss', linewidth=1.5)
    ax.set_title(f'{name}\nFinal Val Loss: {history.history["val_loss"][-1]:.2f}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Hide unused subplot
if len(dl_histories) < len(axes):
    axes[-1].set_visible(False)

plt.suptitle('Training History - All Deep Learning Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '22_dl_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Predicted vs Actual

In [ ]:
# Top 4 DL models: predicted vs actual
top_dl = dl_df.head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.flatten()

for i, name in enumerate(top_dl):
    y_pred = dl_predictions[name]
    r2 = dl_results[name]['R2']
    rmse = dl_results[name]['RMSE']
    
    axes[i].scatter(y_test, y_pred, alpha=0.3, s=8, color='purple')
    axes[i].plot([0, y_test.max()], [0, y_test.max()], 'r--', linewidth=2, label='Perfect')
    axes[i].set_xlabel('Actual AQI')
    axes[i].set_ylabel('Predicted AQI')
    axes[i].set_title(f'{name}\nR²={r2:.4f}, RMSE={rmse:.2f}')
    axes[i].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / '23_dl_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Compare All Models: ML vs Deep Learning

In [ ]:
# Load ML regression results for comparison
ml_results = pd.read_csv(DATA_PROCESSED / 'regression_results.csv', index_col=0)

# Combine ML and DL results
dl_compare = dl_df[['R2', 'RMSE', 'MAE', 'Train_Time_s']].copy()
dl_compare.index = [f'[DL] {n}' for n in dl_compare.index]

ml_compare = ml_results[['R2', 'RMSE', 'MAE', 'Train_Time_s']].copy()
ml_compare.index = [f'[ML] {n}' for n in ml_compare.index]

all_results = pd.concat([ml_compare, dl_compare]).sort_values('R2', ascending=False)

print("=" * 80)
print("COMPLETE MODEL COMPARISON: Traditional ML vs Deep Learning")
print("=" * 80)
print(all_results.round(4).to_string())

# Combined chart
fig, ax = plt.subplots(figsize=(14, 8))
colors = ['#2196F3' if '[ML]' in n else '#9C27B0' for n in all_results.index]
all_results['R2'].plot(kind='barh', ax=ax, color=colors, edgecolor='black', alpha=0.85)
ax.set_xlabel('R² Score')
ax.set_title('All Models: R² Score Comparison (ML vs Deep Learning)')
for i, v in enumerate(all_results['R2']):
    ax.text(max(v + 0.005, 0.01), i, f'{v:.4f}', va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='#2196F3', label='Traditional ML'),
                    Patch(facecolor='#9C27B0', label='Deep Learning')],
          loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '24_all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save DL results
dl_df.to_csv(DATA_PROCESSED / 'deep_learning_results.csv')
all_results.to_csv(DATA_PROCESSED / 'all_model_comparison.csv')

print("Results saved.")
print(f"\nBest overall model: {all_results.index[0]}")
print(f"  R²:   {all_results.iloc[0]['R2']:.4f}")
print(f"  RMSE: {all_results.iloc[0]['RMSE']:.2f}")
print(f"  MAE:  {all_results.iloc[0]['MAE']:.2f}")

print(f"\nBest DL model: {dl_df.index[0]}")
print(f"  R²:   {dl_df.iloc[0]['R2']:.4f}")
print(f"  RMSE: {dl_df.iloc[0]['RMSE']:.2f}")
print(f"  MAE:  {dl_df.iloc[0]['MAE']:.2f}")

## Summary

### Deep Learning Architectures Tested:
1. **Vanilla LSTM** - Single LSTM layer, baseline temporal model
2. **Stacked LSTM** - Two LSTM layers for deeper temporal pattern extraction
3. **Bidirectional LSTM** - Processes sequences forward and backward
4. **Conv1D-LSTM Hybrid** - Conv1D extracts local patterns (3-day windows), LSTM captures long-term trends
5. **GRU** - Gated Recurrent Unit, lightweight LSTM alternative with fewer parameters

### Key Design Decisions:
- **14-day sliding window**: Captures ~2 weeks of pollution history
- **Per-city sequences**: No cross-city leakage in windows
- **Only raw pollutants as features**: Let the network learn its own temporal patterns
- **EarlyStopping + ReduceLROnPlateau**: Prevent overfitting, adaptive learning rate

### ML vs DL:
- Gradient boosting models (LightGBM, CatBoost) with hand-crafted features typically match or outperform DL on tabular data
- DL models are better suited for raw sequence data without feature engineering
- The comparison shows the trade-off between feature engineering effort vs model complexity